# 固定原价整数股五会话：参考与向量化机械身份

冻结五个Qlib来源时钟。每次在完成bar用当时NAV/原价收盘锁股数，下一bar只买指定SOXL或SOXX；恢复前后、半日市及旧多单实例从同一父本状态进行HOLD与EVENT五会话锁定。参考与向量化分别结算并核股数、现金、Tiered、终端NAV和路径回撤。合成零股亦留路径。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.analysis.stage1_capfree_lock_vectorized import compile_lock_price_plane,value_lock_event
from src.backtest.stage1_fixed_share_event import fixed_buy_intent
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_fixed_share_path_parity_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["fixed_order_card"]/"audit.json")==plan["fixed_order_audit_sha256"]
assert sha(root/plan["fixed_order_module"])==plan["fixed_order_module_sha256"]

assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[32723:MainThread](2026-10-30 14:46:01,399) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[32723:MainThread](2026-10-30 14:46:01,400) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[32723:MainThread](2026-10-30 14:46:01,400) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[32723:MainThread](2026-10-30 14:46:01,476) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[32723:MainThread](2026-10-30 14:46:01,477) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[32723:MainThread](2026-10-30 14:46:01,477) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
by_date={};quotes_by_date={};reads=0
for item in plan['cases']:
 day=pd.Timestamp(item['trade_date'])
 if day in by_date:continue
 checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared['selection'],plan=parent_plan)
 by_date[day]=checkpoint
 last=frame.iloc[checkpoint.index+4]
 rawbars=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$raw_close','$price_valid'],start_time=str(day.date())+' 09:31:00',end_time=str(pd.Timestamp(last.trade_date).date())+' 16:00:00',freq='1min')
 parts=[]
 for symbol in ('SOXL','SOXX'):
  x=rawbars.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+v.removeprefix('$') for v in x.columns];parts.append(x)
 quotes_by_date[day]=pd.concat(parts,axis=1).sort_index();reads+=1
assert reads==plan['budget']['qlib_minute_reads']
print('Qlib source sessions',reads)

Qlib source sessions 4


In [3]:
errors={key:0. for key in ('terminal_nav','max_drawdown','cash','quantity','fee')};cases=[];ordinary=None
for item in plan['cases']:
 day=pd.Timestamp(item['trade_date']);checkpoint=by_date[day];quotes=quotes_by_date[day]
 bar_end=pd.Timestamp(str(day.date())+' '+item['bar_end_at']+':00');fill=bar_end+pd.Timedelta(minutes=2)
 late=bar_end>day+pd.Timedelta(hours=10,minutes=32)
 q=checkpoint.state.q.copy();cash=float(checkpoint.state.cash);peak=float(checkpoint.state.peak);fees=deepcopy(checkpoint.state.fees);recovery_state=None;start_at=day+pd.Timedelta(hours=9,minutes=48)
 if late:
  pos=book['native_positions'][day+pd.Timedelta(hours=10,minutes=32)]
  recovery_orders=book['orders'].loc[(book['orders'].trade_date==day)&(book['orders'].slot=='recovery')].to_dict('records')
  recovery_state={'quantities':np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float),'cash':float(pos.get_cash()),'orders':recovery_orders}
  before=quotes.loc[(quotes.index.normalize()==day)&(quotes.index>=start_at)&(quotes.index<=day+pd.Timedelta(hours=10,minutes=32))]
  peak=max(peak,float(np.max(before[['SOXL_close','SOXX_close']].to_numpy(float)@q+cash)))
  q=recovery_state['quantities'].copy();cash=recovery_state['cash']
  for order in recovery_orders:fees.commit(int(order['raw_shares']),stamp=order['stamp'],cap_applied=bool(order['cap_applied']))
  start_at=day+pd.Timedelta(hours=10,minutes=33)
 plane=compile_lock_price_plane(frame.iloc[checkpoint.index:checkpoint.index+5],quotes)
 mark=quotes.loc[bar_end];price=np.array([mark.SOXL_close,mark.SOXX_close],float);nav=float(q@price+cash)
 intent=fixed_buy_intent(symbol=item['asset'],decision_nav=nav,raw_close=float(mark[item['asset']+'_raw_close']),cash=cash,fees=fees,decision_at=bar_end+pd.Timedelta(minutes=1))
 bar=quotes.loc[fill+pd.Timedelta(minutes=1)]
 event={'trade_date':day,'bar_end_at':bar_end,'decision_at':bar_end+pd.Timedelta(minutes=1),'fill_at':fill,'fixed_buy_intent':intent,'adjusted_fill_prices':np.array([bar.SOXL_open,bar.SOXX_open],float),'raw_fill_prices':np.array([bar.SOXL_raw_open,bar.SOXX_raw_open],float)}
 ref=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event,recovery_state=recovery_state,horizon_sessions=5)
 fast=value_lock_event(plane,bar_end_at=bar_end,fill_at=fill,quantities=q,cash=cash,starting_peak=peak,fees=fees,fixed_intent=intent,start_at=start_at)
 assert ref['event_status']==fast['EVENT']['status']==intent['status'] if intent['status']!='ready' else ref['event_status']==fast['EVENT']['status']=='funded'
 assert abs(ref['decision_nav']-fast['decision_nav'])<1e-7
 for arm in ('HOLD','EVENT'):
  x,y=fast[arm],ref[arm]
  local={'terminal_nav':abs(x['terminal_nav']-float(y['path'].iloc[-1].next_open_nav)),'max_drawdown':abs(x['max_drawdown']-y['terminal_available_mark_max_drawdown']),'cash':abs(x['cash']-y['ending_cash']),'quantity':float(np.max(np.abs(x['quantities']-y['ending_quantities']))),'fee':abs(x['fee_usd']-sum(o['total'] for o in y['orders']))}
  for k,v in local.items():errors[k]=max(errors[k],v)
  assert x['monthly_fee_shares']==y['monthly_fee_shares']
  assert [(o['symbol'],o['raw_shares']) for o in x['orders']]==[(o['symbol'],o['raw_shares']) for o in y['orders']]
 other=1 if item['asset']=='SOXL' else 0
 assert fast['EVENT']['quantities'][other]==fast['HOLD']['quantities'][other]
 cases.append({'case':item['reason'],'asset':item['asset'],'status':fast['EVENT']['status'],'raw_shares':intent['raw_shares'],'orders':len(fast['EVENT']['orders']),'other_asset_quantity_change':float(fast['EVENT']['quantities'][other]-fast['HOLD']['quantities'][other]),'source_close':str(quotes.loc[quotes.index.normalize()==day].index.max())})
 if item['reason']=='ordinary early':ordinary=(day,checkpoint,quotes,plane,bar_end,fill,q,cash,peak,fees,bar)
assert len(cases)==5
# A source-independent synthetic no-share quantity remains a HOLD path in both engines.
day,checkpoint,quotes,plane,bar_end,fill,q,cash,peak,fees,bar=ordinary
mark=quotes.loc[bar_end];nav=float(q@np.array([mark.SOXL_close,mark.SOXX_close],float)+cash)
zero=fixed_buy_intent(symbol='SOXX',decision_nav=nav,raw_close=float(mark.SOXX_raw_close),cash=cash,fees=fees,decision_at=bar_end+pd.Timedelta(minutes=1),nominal_beta_increment=1e-8)
assert zero['status']=='decision_no_whole_share'
event={'trade_date':day,'bar_end_at':bar_end,'decision_at':bar_end+pd.Timedelta(minutes=1),'fill_at':fill,'fixed_buy_intent':zero,'adjusted_fill_prices':np.array([bar.SOXL_open,bar.SOXX_open],float),'raw_fill_prices':np.array([bar.SOXL_raw_open,bar.SOXX_raw_open],float)}
ref=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event)
fast=value_lock_event(plane,bar_end_at=bar_end,fill_at=fill,quantities=q,cash=cash,starting_peak=peak,fees=fees,fixed_intent=zero,start_at=day+pd.Timedelta(hours=9,minutes=48))
assert ref['event_status']==fast['EVENT']['status']=='decision_no_whole_share'
assert not ref['EVENT']['orders'] and not fast['EVENT']['orders']
assert fast['EVENT']['terminal_nav']==fast['HOLD']['terminal_nav'] and fast['EVENT']['max_drawdown']==fast['HOLD']['max_drawdown']
assert errors['terminal_nav']<=plan['maximum_nav_error_usd'] and errors['max_drawdown']<=plan['maximum_risk_error']
assert errors['cash']==errors['quantity']==errors['fee']==0
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'order_module_sha256':plan['fixed_order_module_sha256'],'reference_module_sha256':sha(root/plan['reference_module']),'vector_module_sha256':sha(root/plan['vector_module']),'cases':cases,'max_parity_errors':errors,'synthetic_zero_share_path_identity':True,'qlib_minute_reads':reads,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'fixed_share_five_session_reference_vector_mechanical_parity_pass'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_fixed_share_path_parity_20261030",
  "plan_sha256": "e87a5b625fb280ce2d05a2391945db7bc38ce9daa78f687c90d5de8d9fc9df13",
  "order_module_sha256": "5bf69f1d17e266d869ce13ba6511ea8b1675b1b24e3d382fe4285f20d3deefe7",
  "reference_module_sha256": "7c875695eba97351c0f20eac16a7e89c887a470b8d297c13e43cdfc6d8c4d79b",
  "vector_module_sha256": "c813c720e56891ad5c75bf38ea454ad32c5615449fbc63644fe18a1c20b21f32",
  "cases": [
    {
      "case": "first source-only old extra SOXL sale",
      "asset": "SOXX",
      "status": "funded",
      "raw_shares": 59,
      "orders": 1,
      "other_asset_quantity_change": 0.0,
      "source_close": "2018-10-15 16:00:00"
    },
    {
      "case": "13:00 half-day",
      "asset": "SOXX",
      "status": "funded",
      "raw_shares": 60,
      "orders": 1,
      "other_asset_quantity_change": 0.0,
      "source_close": "2018-11-23 13:00:00"
    },
    {
      "case": "ordinary early",
      "asset": "SOXX",
      "status": "funded"

## 中文结论

SOXX历史Goal未通过。固定决策时原价整数股订单接入逐事件与向量化五会话HOLD/EVENT路径，五个固定Qlib普通早盘、13:00半日市、恢复前/后和旧多单例的终端NAV、可用mark最大回撤、股数、现金、Tiered费用及月度计数逐项最大差0；每例只有指定一笔买单，未动资产股数差0。合成决策时不足一股返回`decision_no_whole_share`，两支终端与风险一致。初次启动在读Qlib前因计划缺`checkpoint_code`键停止，原计划和收据保留，机械补键后4次预定来源读取完成。此卡不包含全日历新动作标签或信号，不可挪用旧完整权重版价值。下批先冻固定股数全年来源支持和成熟门，再生成同状态Qlib Recorder标签及信息门。